In [ ]:
def PCDTWMSAlign(FASTALoc, PCProp1='Mass', PCProp2='HydroPho', n_jobs=-1):

    # Read the input sequences from a fasta file
    if isinstance(FASTALoc, str):
        records = list(SeqIO.parse(FASTALoc, "fasta"))
        seqs = [str(record.seq) for record in records]
        names = [record.id for record in records]
    elif isinstance(FASTALoc, list):
        seqs = [str(seq) for seq in FASTALoc]
        names = [f"seq{i+1}" for i in range(len(seqs))]
    else:
        raise ValueError("FASTALoc must be a file path (str) or a list of sequences")

    seqs_len = len(seqs)
    if seqs_len < 2:
        raise ValueError("At least two sequences are required for MSA")

    # Compute pairwise distance matrix to find the star sequence
    asym_mat = np.zeros((seqs_len, seqs_len), dtype=np.float64)
    indices = [(i, j) for i in range(seqs_len) for j in range(i + 1, seqs_len)]
    distances = Parallel(n_jobs=n_jobs)(
        delayed(lambda i, j: PCDTWDist(seqs[i], seqs[j], PCProp1=PCProp1, PCProp2=PCProp2))(i, j)
        for i, j in indices
    )
    for k, (i, j) in enumerate(indices):
        asym_mat[i, j] = distances[k]
        asym_mat[j, i] = distances[k] 

    # Make the distance matrix symetric
    sym_matrix = (asym_mat + asym_mat.T) / 2
    np.fill_diagonal(sym_matrix, 0)

    # Find the star sequence index
    star_idx = np.argmin(np.sum(sym_matrix, axis=1))
    star_seq = seqs[star_idx]
    non_star_seqs = seqs[:star_idx] + seqs[star_idx + 1:]
    non_star_indices = list(range(star_idx)) + list(range(star_idx + 1, seqs_len))

    # Perform pairwise alignments to star sequence
    alignments = Parallel(n_jobs=n_jobs)(
        delayed(lambda seq: (
            PCDTWPWAlign(star_seq, seq, PCProp1=PCProp1, PCProp2=PCProp2, Penalty=0, Window=3, GAP='Gap')
        ))(seq) for seq in non_star_seqs
    )
    alignments = [
        (aln['Seq1AlignedString'].replace('\n', '').replace('\r', ''),
         aln['Seq2AlignedString'].replace('\n', '').replace('\r', ''))
        for aln in alignments
    ]

    # Initialize MSA with the first pairwise alignment
    star0, other0 = alignments[0]
    msa = np.array([list(star0), list(other0)], dtype=np.dtype('U1'))
    seq_counter = 1
    names_dict = {0: 'star', 1: 'seq1'}

    # Merge the remaining alignments
    for aln_star, aln_other in alignments[1:]:
        seq_counter += 1
        aln_star = np.array(list(aln_star), dtype=np.dtype('U1'))
        aln_other = np.array(list(aln_other), dtype=np.dtype('U1'))
        
        # Pre-allocate new MSA with enough space
        max_len = msa.shape[1] + len(aln_star)
        new_msa = np.full((msa.shape[0] + 1, max_len), '-', dtype=np.dtype('U1'))
        new_msa[:msa.shape[0], :msa.shape[1]] = msa
        i = j = k = 0

        # Merge the sequences
        while i < msa.shape[1] or j < len(aln_star):
            a = msa[0, i] if i < msa.shape[1] else None
            b = aln_star[j] if j < len(aln_star) else None

            if a is not None and b is not None and a == b:
                new_msa[:-1, k] = msa[:, i]
                new_msa[-1, k] = aln_other[j]
                i += 1
                j += 1
            elif a == '-' and (b != '-' or b is None):
                new_msa[:-1, k] = msa[:, i]
                new_msa[-1, k] = '-'
                i += 1
            elif b == '-' and (a != '-' or a is None):
                new_msa[:-1, k] = '-'
                new_msa[-1, k] = aln_other[j]
                j += 1
            elif a is None and b is not None:
                new_msa[:-1, k] = '-'
                new_msa[-1, k] = aln_other[j]
                j += 1
            elif b is None and a is not None:
                new_msa[:-1, k] = msa[:, i]
                new_msa[-1, k] = '-'
                i += 1
            k += 1

        # Trim excess columns
        msa = new_msa[:, :k]
        names_dict[msa.shape[0] - 1] = f"seq{seq_counter}"

    # Prepare a list of aligned sequences as output
    keys_order = ['star'] + [f"seq{i+1}" for i in range(seq_counter)]
    aligned_seqs = [''.join(msa[i]) for i in range(msa.shape[0])]
    name_mapping = {names_dict[i]: names[star_idx if i == 0 else non_star_indices[i-1]] for i in range(msa.shape[0])}
    
    # Format the output string so it is in fasta format
    output_str = ""
    for key in keys_order:
        output_str += f">{name_mapping[key]}\n{aligned_seqs[keys_order.index(key)]}\n"

    return aligned_seqs, output_str
